In [1]:
from pathlib import Path
import pandas as pd
import nd2

images_dir = Path("/Users/kelpschdj/Documents/DataTecnica/TTU/Particle_tracking/Data/Sphere/220725_i11w-hT-M33-I76_sg1035_d10sphere")
images = list(sorted(images_dir.glob('*.nd2')))

image_index = 12

image_path = images[image_index]
print(image_path)
   
img_array = nd2.imread(image_path)
print(img_array.shape)
print(type(img_array))

tracks_df = pd.read_csv("/Users/kelpschdj/Documents/DataTecnica/TTU/Particle_tracking/Data/Sphere/220725_i11w-hT-M33-I76_sg1035_d10sphere/all_valid_20250605.csv")
tracks_df = tracks_df[tracks_df["image_name"] == image_path.name]

/Users/kelpschdj/Documents/DataTecnica/TTU/Particle_tracking/Data/Sphere/220725_i11w-hT-M33-I76_sg1035_d10sphere/sg100_Well7_1025.nd2
(33, 4, 2048, 2048)
<class 'numpy.ndarray'>


In [2]:
import napari

viewer = napari.Viewer()

viewer.add_image(img_array[:, 0], 
                 name='Halo-TDP-43')

viewer.add_image(img_array[:, 1], 
                 name='LYSOSOME')

viewer.add_image(img_array[:, 2], 
                 name='MITOCHONDRIA')

viewer.add_image(img_array[:, 3], 
                 name='BFP')

<Image layer 'BFP' at 0x1772c7ed0>

In [3]:
import os
import numpy as np
import nd2
import imageio.v3 as iio
from PIL import Image, ImageDraw

# ---------------------------
# Normalization + rendering helpers
# ---------------------------

def normalize_crop_percentile(crop, lower=2, upper=98):
    """
    Robust per-channel percentile-based normalization.
    crop: (T, C, h, w) float/uint -> uint8 (T, C, h, w)
    Percentiles computed over ALL frames in the crop per channel.
    """
    T, C, h, w = crop.shape
    crop_norm = np.zeros((T, C, h, w), dtype=np.uint8)

    for c in range(C):
        channel_data = crop[:, c].reshape(-1).astype(np.float32)
        p_low, p_high = np.percentile(channel_data, [lower, upper])

        if (p_high - p_low) < 1e-3:
            crop_norm[:, c] = 0
        else:
            norm = (crop[:, c].astype(np.float32) - p_low) / (p_high - p_low)
            norm = np.clip(norm * 255.0, 0, 255)
            crop_norm[:, c] = norm.astype(np.uint8)

    return crop_norm

def _to_rgb(gray_u8):
    """(H,W) uint8 -> (H,W,3) uint8 grayscale RGB."""
    return np.stack([gray_u8, gray_u8, gray_u8], axis=-1)

def _apply_pseudocolor(gray_u8, rgb_color):
    """
    gray_u8: (H,W) uint8
    rgb_color: (R,G,B) in 0-255
    returns: (H,W,3) uint8
    """
    g = gray_u8.astype(np.float32) / 255.0
    color = np.array(rgb_color, dtype=np.float32) / 255.0
    out = g[..., None] * color[None, None, :]
    return (np.clip(out, 0, 1) * 255).astype(np.uint8)

def _merge_rgb(rgb_list):
    """Additive merge with clamp."""
    if len(rgb_list) == 0:
        raise ValueError("rgb_list is empty; nothing to merge.")
    acc = np.zeros_like(rgb_list[0], dtype=np.float32)
    for im in rgb_list:
        acc += im.astype(np.float32)
    return np.clip(acc, 0, 255).astype(np.uint8)

def _draw_overlays(rgb_u8, center_xy, track_xy,
                   circle_r=6, line_w=2,
                   circle_color=(255, 0, 0),      # red
                   line_color=(255, 255, 0)):     # yellow
    """
    Draw a circle at center and a polyline track.
    center_xy: (x,y) pixel coords in the current image
    track_xy: list[(x,y)] points (same coords)
    """
    im = Image.fromarray(rgb_u8)
    dr = ImageDraw.Draw(im)

    cx, cy = center_xy
    dr.ellipse((cx - circle_r, cy - circle_r, cx + circle_r, cy + circle_r),
               outline=circle_color, width=line_w)

    if len(track_xy) >= 2:
        dr.line(track_xy, fill=line_color, width=line_w, joint="curve")

    return np.array(im, dtype=np.uint8)

def _resize_nn(rgb_u8, scale):
    """Integer upscale via nearest-neighbor (keeps pixels crisp)."""
    if scale == 1:
        return rgb_u8
    im = Image.fromarray(rgb_u8)
    w, h = im.size
    im = im.resize((w * scale, h * scale), resample=Image.Resampling.NEAREST)
    return np.array(im, dtype=np.uint8)

def apply_gamma_u8(gray_u8, gamma=0.8):
    """
    Apply display gamma to uint8 grayscale image.
    gamma < 1  → brighten dim structures
    gamma > 1  → darken midtones
    """
    g = gray_u8.astype(np.float32) / 255.0
    g = np.power(g, gamma)
    return (np.clip(g, 0, 1) * 255).astype(np.uint8)



# ---------------------------
# Main writer
# ---------------------------

def save_track_gifs(
    img_array,               # (T, C, Y, X)
    track_df,                # must contain: particle, frame, y, x
    image_name,
    output_dir,
    padding=40,              # <-- bbox padding around entire track
    fps=10,
    upscale=2,               # 2 is usually a nice default
    norm_lower=2,
    norm_upper=98,
    channel_colors=None,     # dict: {0:(...), 1:(...), 2:(...)}
    composite_channels=(0, 1, 2),
    circle_r=6,
    line_w=2,
    gamma = 0.8,
    only_track_frames=True,  # write only frames where track exists
):
    output_dir = str(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    if img_array.ndim != 4:
        raise ValueError(f"Expected img_array shape (T,C,Y,X), got {img_array.shape}")

    T, C, Y, X = img_array.shape

    # Defaults requested
    if channel_colors is None:
        channel_colors = {
            0: (255, 0, 255),  # magenta
            1: (255, 0, 0),    # red
            2: (0, 255, 0),    # green
        }

    base_name = os.path.splitext(image_name)[0].replace(" ", "_")

    df = track_df.copy()
    df["frame"] = df["frame"].astype(int)
    df = df.sort_values(["particle", "frame"])

    duration = 1.0 / fps

    for pid, group in df.groupby("particle"):
        group = group.sort_values("frame")

        # Track-level bbox in full-image coords
        y_min, y_max = int(np.floor(group["y"].min())), int(np.ceil(group["y"].max()))
        x_min, x_max = int(np.floor(group["x"].min())), int(np.ceil(group["x"].max()))

        # Apply padding (your 40 px requirement)
        y1 = max(y_min - padding, 0)
        y2 = min(y_max + padding, Y)
        x1 = max(x_min - padding, 0)
        x2 = min(x_max + padding, X)

        if y2 <= y1 or x2 <= x1:
            continue

        # Crop all frames then normalize per-channel over all timepoints
        crop = img_array[:, :, y1:y2, x1:x2]  # (T, C, h, w)
        crop_norm = normalize_crop_percentile(crop, lower=norm_lower, upper=norm_upper)
        h, w = crop_norm.shape[2], crop_norm.shape[3]

        # Map frame -> center (in crop coords)
        centers = {
            int(r["frame"]): (float(r["x"]) - x1, float(r["y"]) - y1)
            for _, r in group.iterrows()
            if 0 <= int(r["frame"]) < T
        }

        # Progressive track line (grows over time)
        track_by_frame = {}
        running = []
        for t in sorted(centers.keys()):
            running.append(centers[t])
            track_by_frame[t] = running.copy()

        # Which frames to write
        if only_track_frames:
            t_list = np.array(sorted(centers.keys()), dtype=int)
        else:
            t_list = np.arange(T, dtype=int)

        if len(t_list) == 0:
            continue

        # Frame buffers
        per_ch_raw = [[] for _ in range(C)]
        per_ch_ann = [[] for _ in range(C)]
        comp_raw = []
        comp_ann = []

        def scale_pt(p):
            return (p[0] * upscale, p[1] * upscale)

        for t in t_list:
            center = centers.get(int(t), None)
            pts_now = track_by_frame.get(int(t), [])

            pseudo_list = []

            for c in range(C):
                gray = crop_norm[t, c]  
                # gray = apply_gamma_u8(gray, gamma=gamma)  

                # Per-channel grayscale
                rgb_raw = _to_rgb(gray)
                rgb_raw = _resize_nn(rgb_raw, upscale)

                rgb_ann = rgb_raw
                if center is not None:
                    rgb_ann = _draw_overlays(
                        rgb_raw.copy(),
                        center_xy=scale_pt(center),
                        track_xy=[scale_pt(p) for p in pts_now],
                        circle_r=circle_r * upscale,
                        line_w=max(1, line_w * upscale),
                        circle_color=(255, 0, 0),      # red circle
                        line_color=(255, 255, 0),      # yellow line
                    )

                per_ch_raw[c].append(rgb_raw)
                per_ch_ann[c].append(rgb_ann)

                # Composite uses only selected channels
                if c in composite_channels:
                    color = channel_colors.get(c, (255, 255, 255))
                    rgb_pc = _apply_pseudocolor(gray, color)
                    rgb_pc = _resize_nn(rgb_pc, upscale)
                    pseudo_list.append(rgb_pc)

            merged = _merge_rgb(pseudo_list) if len(pseudo_list) else np.zeros((h*upscale, w*upscale, 3), dtype=np.uint8)
            comp_raw.append(merged)

            merged_ann = merged
            if center is not None:
                merged_ann = _draw_overlays(
                    merged.copy(),
                    center_xy=scale_pt(center),
                    track_xy=[scale_pt(p) for p in pts_now],
                    circle_r=circle_r * upscale,
                    line_w=max(1, line_w * upscale),
                    circle_color=(255, 0, 0),      # red circle
                    line_color=(255, 255, 0),      # yellow line
                )
            comp_ann.append(merged_ann)

        # Write GIFs
        for c in range(C):
            fn_raw = os.path.join(output_dir, f"{base_name}_track{pid}_ch{c}_raw.gif")
            fn_ann = os.path.join(output_dir, f"{base_name}_track{pid}_ch{c}_annot.gif")
            iio.imwrite(fn_raw, per_ch_raw[c], duration=duration, loop=0)
            iio.imwrite(fn_ann, per_ch_ann[c], duration=duration, loop=0)

        fn_comp_raw = os.path.join(output_dir, f"{base_name}_track{pid}_composite_raw.gif")
        fn_comp_ann = os.path.join(output_dir, f"{base_name}_track{pid}_composite_annot.gif")
        iio.imwrite(fn_comp_raw, comp_raw, duration=duration, loop=0)
        iio.imwrite(fn_comp_ann, comp_ann, duration=duration, loop=0)

        print(f"[track {pid}] wrote gifs to: {output_dir}")


In [4]:
# Composite colors and channels (exclude ch4)
channel_colors = {
    0: (255, 0, 255),  # magenta
    1: (255, 0, 0),    # red
    2: (0, 255, 0),    # green
}
composite_channels = (0, 1, 2)
output_dir = "track_gifs"

# Write GIFs
save_track_gifs(
    img_array=img_array,
    track_df=tracks_df,
    image_name=image_path.name,
    output_dir=output_dir,
    padding=40,
    fps=10,
    upscale=2,
    norm_lower=2,
    norm_upper=98,
    channel_colors=channel_colors,
    composite_channels=composite_channels,
    circle_r=8,
    line_w=2,
    gamma = 1.2,
    only_track_frames=True,
)



[track 41] wrote gifs to: track_gifs
[track 55] wrote gifs to: track_gifs
[track 101] wrote gifs to: track_gifs
[track 110] wrote gifs to: track_gifs
[track 129] wrote gifs to: track_gifs
[track 406] wrote gifs to: track_gifs
[track 776] wrote gifs to: track_gifs
[track 1116] wrote gifs to: track_gifs
[track 1118] wrote gifs to: track_gifs
[track 1626] wrote gifs to: track_gifs
[track 1684] wrote gifs to: track_gifs


In [5]:
from skimage.filters import gaussian
from scipy.ndimage import median_filter
import napari
import numpy as np

#median project and smooth the image
mean_proj = np.mean(img_array[:, 0], axis=0) 
#smoothed_background = gaussian(mean_proj, sigma=50)

#generate a median filter of the image
filtered = median_filter(mean_proj, size=10)

# Preprocess: background subtraction
preprocessed_frames = []

for frame in img_array[:, 0]:
    foreground = frame - filtered
    foreground = np.clip(foreground, 0, None).astype(np.uint16)
    preprocessed_frames.append(foreground)

preprocessed_stack = np.stack(preprocessed_frames)

viewer = napari.Viewer()

viewer.add_image(img_array[:, 0], 
                 name='Halo-TDP-43')

viewer.add_image(preprocessed_stack, 
                 name='PROCESSED Halo-TDP-43')

viewer.add_image(img_array[:, 1], 
                 name='LYSOSOME')

viewer.add_image(img_array[:, 2], 
                 name='MITOCHONDRIA')

viewer.add_image(img_array[:, 3], 
                 name='BFP')

<Image layer 'BFP' at 0x33360e990>

In [ ]:
def save_channel_gifs(
    img_array,              # (T, C, Y, X)
    preprocessed_stack=None,# (T, Y, X) optional
    image_name="movie",
    output_dir="gifs",
    fps=10,
    upscale=1,
    norm_lower=2,
    norm_upper=98,
    gamma=0.8,
):
    os.makedirs(output_dir, exist_ok=True)

    T, C, Y, X = img_array.shape
    duration = 1.0 / fps
    base = os.path.splitext(image_name)[0]

    # ---- Build channel dictionary ----
    channel_dict = {f"ch{c}": img_array[:, c] for c in range(C)}

    if preprocessed_stack is not None:
        channel_dict["processed"] = preprocessed_stack

    # ---- Process each channel ----
    for name, stack in channel_dict.items():

        # stack: (T, Y, X) → fake channel dimension so we can reuse normalize_crop_percentile
        stack4d = stack[:, None, :, :]
        stack_norm = normalize_crop_percentile(
            stack4d,
            lower=norm_lower,
            upper=norm_upper
        )[:, 0]  # back to (T,Y,X)

        frames = []

        for t in range(T):
            gray = stack_norm[t]
            gray = apply_gamma_u8(gray, gamma=gamma)

            rgb = _to_rgb(gray)
            rgb = _resize_nn(rgb, upscale)

            frames.append(rgb)

        out_path = os.path.join(output_dir, f"{base}_{name}.gif")
        iio.imwrite(out_path, frames, duration=duration, loop=0)

        print("Wrote:", out_path)

save_channel_gifs(
    img_array=img_array,
    preprocessed_stack=preprocessed_stack,
    image_name="sg100_Well7_1025_fullframe",
    output_dir="channel_gifs",
    fps=10,
    upscale=1,
    norm_upper=99.99,
    gamma=0.98,
)

In [ ]:
def save_channel_gifs(
    img_array,               # (T, C, Y, X)
    preprocessed_stack=None, # (T, Y, X) optional
    image_name="movie",
    output_dir="gifs",
    fps=10,
    upscale=1,
    norm_lower=2,
    norm_upper=98,
    gamma=0.8,
    roi=None,                # (x1, x2, y1, y2) in pixel coords
):
    os.makedirs(output_dir, exist_ok=True)

    if img_array.ndim != 4:
        raise ValueError(f"Expected img_array shape (T,C,Y,X), got {img_array.shape}")

    T, C, Y, X = img_array.shape
    duration = 1.0 / fps
    base = os.path.splitext(image_name)[0]

    # ---- ROI bounds ----
    if roi is None:
        x1, x2, y1, y2 = 0, X, 0, Y
        roi_tag = "full"
    else:
        x1, x2, y1, y2 = roi
        x1 = max(0, int(x1)); x2 = min(X, int(x2))
        y1 = max(0, int(y1)); y2 = min(Y, int(y2))
        if x2 <= x1 or y2 <= y1:
            raise ValueError(f"Invalid ROI after clipping: {(x1, x2, y1, y2)}")
        roi_tag = f"x{x1}-{x2}_y{y1}-{y2}"

    # ---- Build channel dictionary (cropped) ----
    channel_dict = {f"ch{c}": img_array[:, c, y1:y2, x1:x2] for c in range(C)}
    if preprocessed_stack is not None:
        channel_dict["processed"] = preprocessed_stack[:, y1:y2, x1:x2]

    # ---- Process each channel ----
    for name, stack in channel_dict.items():
        # stack: (T, Y, X) → fake channel dimension so we can reuse normalize_crop_percentile
        stack4d = stack[:, None, :, :]
        stack_norm = normalize_crop_percentile(
            stack4d,
            lower=norm_lower,
            upper=norm_upper
        )[:, 0]  # back to (T,Y,X)

        frames = []
        for t in range(T):
            gray = stack_norm[t]
            gray = apply_gamma_u8(gray, gamma=gamma)

            rgb = _to_rgb(gray)
            rgb = _resize_nn(rgb, upscale)
            frames.append(rgb)

        out_path = os.path.join(output_dir, f"{base}_{roi_tag}_{name}.gif")
        iio.imwrite(out_path, frames, duration=duration, loop=0)
        print("Wrote:", out_path)

save_channel_gifs(
    img_array=img_array,
    preprocessed_stack=preprocessed_stack,
    image_name="sg100_Well7_1025_fullframe",
    output_dir="channel_gifs",
    fps=10,
    upscale=1,
    norm_upper=99.99,
    gamma=0.98,
    roi=(700, 1100, 950, 1050),  # (x1, x2, y1, y2)
)


In [ ]:
from PIL import Image

def _overlay_png(rgb_u8, center_xy, png_path,
                 scale=1.0,
                 anchor="center",
                 opacity=1.0):
    """
    Paste an RGBA PNG onto an RGB frame.
    rgb_u8: (H,W,3) uint8
    center_xy: (x,y) in pixels (same coordinate space as rgb_u8)
    png_path: path to a PNG with transparency (RGBA)
    scale: resize factor applied to the PNG
    anchor: 'center' or 'topleft'
    opacity: 0..1 multiplier on PNG alpha
    """
    base = Image.fromarray(rgb_u8).convert("RGBA")

    sprite = Image.open(png_path).convert("RGBA")
    if scale != 1.0:
        sw, sh = sprite.size
        sprite = sprite.resize(
            (max(1, int(round(sw * scale))), max(1, int(round(sh * scale)))),
            resample=Image.Resampling.LANCZOS
        )

    if opacity < 1.0:
        r, g, b, a = sprite.split()
        a = a.point(lambda v: int(v * opacity))
        sprite = Image.merge("RGBA", (r, g, b, a))

    x, y = center_xy
    if anchor == "center":
        x = int(round(x - sprite.size[0] / 2))
        y = int(round(y - sprite.size[1] / 2))
    else:
        x = int(round(x))
        y = int(round(y))

    base.alpha_composite(sprite, (x, y))
    return np.array(base.convert("RGB"), dtype=np.uint8)

def save_track_gifs(
    img_array,
    track_df,
    image_name,
    output_dir,
    padding=40,
    fps=10,
    upscale=2,
    norm_lower=2,
    norm_upper=98,
    channel_colors=None,
    composite_channels=(0, 1, 2),
    line_w=2,
    gamma=0.8,
    only_track_frames=True,

    # NEW:
    marker_png_path=None,     # path to your small PNG (with transparency)
    marker_scale=1.0,         # relative to the PNG's native size
    marker_opacity=1.0,       # 0..1
    draw_track_line=True,     # if you ever want to disable the yellow track
):
    output_dir = str(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    if img_array.ndim != 4:
        raise ValueError(f"Expected img_array shape (T,C,Y,X), got {img_array.shape}")

    if marker_png_path is None:
        raise ValueError("marker_png_path is required for the PNG-marker version.")

    T, C, Y, X = img_array.shape

    if channel_colors is None:
        channel_colors = {
            0: (255, 0, 255),  # magenta
            1: (255, 0, 0),    # red
            2: (0, 255, 0),    # green
        }

    base_name = os.path.splitext(image_name)[0].replace(" ", "_")

    df = track_df.copy()
    df["frame"] = df["frame"].astype(int)
    df = df.sort_values(["particle", "frame"])

    duration = 1.0 / fps

    for pid, group in df.groupby("particle"):
        group = group.sort_values("frame")

        y_min, y_max = int(np.floor(group["y"].min())), int(np.ceil(group["y"].max()))
        x_min, x_max = int(np.floor(group["x"].min())), int(np.ceil(group["x"].max()))

        y1 = max(y_min - padding, 0)
        y2 = min(y_max + padding, Y)
        x1 = max(x_min - padding, 0)
        x2 = min(x_max + padding, X)

        if y2 <= y1 or x2 <= x1:
            continue

        crop = img_array[:, :, y1:y2, x1:x2]
        crop_norm = normalize_crop_percentile(crop, lower=norm_lower, upper=norm_upper)
        h, w = crop_norm.shape[2], crop_norm.shape[3]

        centers = {
            int(r["frame"]): (float(r["x"]) - x1, float(r["y"]) - y1)
            for _, r in group.iterrows()
            if 0 <= int(r["frame"]) < T
        }

        track_by_frame = {}
        running = []
        for t in sorted(centers.keys()):
            running.append(centers[t])
            track_by_frame[t] = running.copy()

        if only_track_frames:
            t_list = np.array(sorted(centers.keys()), dtype=int)
        else:
            t_list = np.arange(T, dtype=int)

        if len(t_list) == 0:
            continue

        comp_raw = []
        comp_ann = []

        def scale_pt(p):
            return (p[0] * upscale, p[1] * upscale)

        for t in t_list:
            center = centers.get(int(t), None)
            pts_now = track_by_frame.get(int(t), [])

            pseudo_list = []
            for c in range(C):
                gray = crop_norm[t, c]
                # gray = apply_gamma_u8(gray, gamma=gamma)

                if c in composite_channels:
                    color = channel_colors.get(c, (255, 255, 255))
                    rgb_pc = _apply_pseudocolor(gray, color)
                    rgb_pc = _resize_nn(rgb_pc, upscale)
                    pseudo_list.append(rgb_pc)

            merged = _merge_rgb(pseudo_list) if len(pseudo_list) else np.zeros((h*upscale, w*upscale, 3), dtype=np.uint8)
            comp_raw.append(merged)

            merged_ann = merged.copy()
            if center is not None:
                # draw track line (optional)
                if draw_track_line and len(pts_now) >= 2:
                    merged_ann = _draw_overlays(
                        merged_ann,
                        center_xy=scale_pt(center),
                        track_xy=[scale_pt(p) for p in pts_now],
                        circle_r=0,                 # won't matter
                        line_w=max(1, line_w * upscale),
                        circle_color=(0, 0, 0),     # ignored
                        line_color=(255, 255, 0),
                    )

                # overlay PNG marker (replaces circle)
                merged_ann = _overlay_png(
                    merged_ann,
                    center_xy=scale_pt(center),
                    png_path=marker_png_path,
                    scale=marker_scale * upscale,   # keeps marker visually consistent when upscaling
                    opacity=marker_opacity,
                    anchor="center",
                )

            comp_ann.append(merged_ann)

        fn_comp_raw = os.path.join(output_dir, f"{base_name}_track{pid}_composite_raw.gif")
        fn_comp_ann = os.path.join(output_dir, f"{base_name}_track{pid}_composite_pngmarker.gif")
        iio.imwrite(fn_comp_raw, comp_raw, duration=duration, loop=0)
        iio.imwrite(fn_comp_ann, comp_ann, duration=duration, loop=0)

        print(f"[track {pid}] wrote composite gifs to: {output_dir}")

save_track_gifs(
    img_array=img_array,
    track_df=tracks_df.query("particle == 41"),
    image_name="my_dumb_movie.nd2",
    output_dir="dumb_gifs",
    marker_png_path="/Users/kelpschdj/Documents/DataTecnica/TTU/Track2GIF/kelpsch_headshot.png",  # must be RGBA PNG
    marker_scale=0.02,   # tune this
    marker_opacity=1.0,
)

